# 04 · GeoStructureNet (Experiment E)

One ablation axis per run, parameter counts compared in each experiment's `architecture.csv`:

| run | change vs E1 |
|---|---|
| E1 `e1_gsn_parity` | none - GeoStructureNet configured as the gated dual encoder (must match B0) |
| E2 | modality-wise attention fusion |
| E3 | cross-scale gated fusion |
| E4 | dilated long-range bottleneck (Fourier = separate copy, research only) |
| E5 | gated coarse context branch (strain / seismicity) |

Also: gate activation maps and a shifted-tile consistency probe (tile-edge artefacts).

In [ ]:
# --- session bootstrap (idempotent; identical in notebooks 00-07) ---------------------------------
REPO = 'mybenkhadda/GEMS---drivendata'
COMMIT = 'main'
DRIVE_ROOT = '/content/drive/MyDrive/GEMS_2'
import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')
token = ''
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN') or ''
except Exception:
    pass
url = f'https://{token}@github.com/{REPO}.git' if token else f'https://github.com/{REPO}.git'
if not os.path.exists('/content/GEMS/.git'):
    subprocess.run(['git', 'clone', '-q', url, '/content/GEMS'], check=True)
subprocess.run(['git', '-C', '/content/GEMS', 'fetch', '-q', 'origin'], check=True)
subprocess.run(['git', '-C', '/content/GEMS', 'checkout', '-q', COMMIT], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '/content/GEMS[colab]'], check=True)
if '/content/GEMS' not in sys.path:
    sys.path.insert(0, '/content/GEMS')
from gems_bench.runtime import colab
colab.configure(drive_root=DRIVE_ROOT)
colab.stage_core(); colab.stage_legacy();

In [ ]:
# Prepare all variants first (cheap): compare parameter counts before spending GPU time
import pandas as pd
from gems_bench.experiment import Experiment
C = '/content/GEMS/configs/experiments/'
names = ['e1_gsn_parity', 'e2_gsn_attention', 'e3_gsn_crossscale', 'e4_gsn_dilated', 'e5_gsn_context']
E = {n: Experiment.from_yaml(C + n + '.yaml') for n in names}
for e in E.values():
    e.prepare()
pd.concat([e.rd.load_table('architecture').assign(experiment=n) for n, e in E.items()])

In [ ]:
for n, e in E.items():
    e.run_all()
    if e.rd.has_table('promotion_decisions'):
        display(e.rd.load_table('promotion_decisions'))

In [ ]:
# Gate activation maps (terrain vs potential-field reliance) and shifted-tile consistency for one variant
import numpy as np
import matplotlib.pyplot as plt
from gems_bench.cnn import device_of
from gems_bench.phase4_training import to_tensor_stack, pooled_context, context_crop, _forward
from gems_bench.p3 import cnn3
from gems_bench.training.checkpoint import CheckpointManager
from gems_bench.training.trainer import load_unit_model
from gems_bench.training.cv import prepare_stack5
from gems_bench.inference.full_raster import shifted_tile_consistency
e = E['e2_gsn_attention']
ctx, dev = e.ctx, device_of()
spec, model = load_unit_model(CheckpointManager(e.rd.checkpoints, 'cv_fold0'), dev)
g = to_tensor_stack(prepare_stack5(ctx, spec), dev)
gctx = pooled_context(g, spec.context_factor) if spec.context else None
rr, cc = np.nonzero(ctx.data.labels & ctx.folds.val(0))
r0, c0 = int(rr[len(rr) // 2]) - 128, int(cc[len(cc) // 2]) - 128
x = g[:, r0:r0 + 256, c0:c0 + 256][None].to(dev).float()
c = context_crop(gctx, r0, c0, 256, spec.context_factor)[None].to(dev).float() if gctx is not None else None
model.eval(); _forward(model, x, c, spec, dev, cnn3._amp_dtype(spec, dev))
gate = model.gates()[0]
fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow(x[0, 0].cpu(), cmap='gray'); ax[0].set_title('detrended elevation')
ax[1].imshow(gate[0, 0].float().cpu(), cmap='Blues', vmin=0, vmax=1); ax[1].set_title('level-0 weight on terrain branch')
e.rd.save_fig(fig, 'gate_activation_example')
print(shifted_tile_consistency(model, g, (r0, r0 + 512, c0, c0 + 512), spec, dev, 64, gctx))